# panvram quickstart - the HPRC cohort (558 assemblies) resident on one GPU

Downloads everything it needs (no Google Drive): the panvram package, T2T-CHM13v2.0 from NCBI (md5 checked) and the
refrel3 v1 archives listed in `MANIFEST.tsv`, each checked against the manifest's SHA-256 before use. Then opens the
cohort on the GPU, samples windows, fetches regions and decodes every assembly in full, checking each FASTA XXH3
against the manifest (no source FASTA needed).

**Before running:** set `ARCHIVE_BASE_URL` to the published dataset location (placeholder until the dataset record
exists) and `PANVRAM_SRC` to the package source. Needs a CUDA GPU of sm_80 or newer; q4k needs ~12.3 GB of disk for
the archives + 3.2 GB for T2T, and ~16.7 GB of GPU memory in the default form (11.7 GB with both compact forms).
Set `N` to a smaller number for a quick try (the first N rows of the manifest).

Data use: HPRC data are public domain; do not sell them, do not attempt participant re-identification, acknowledge
"the Human Pangenome Reference Consortium (BioProject ID: PRJNA730823) and its funder, the National Human Genome
Research Institute (NHGRI)" (see DATASET.md).

In [ ]:
# 1. Settings
import os
ARCHIVE_BASE_URL = "https://REPLACE-WITH-DATASET-RECORD/files"   # placeholder: where <name>.q4k.rr3 are published
PANVRAM_SRC = "https://REPLACE-WITH-PANVRAM-RELEASE/panvram-1.0.0.tar.gz"  # placeholder: the package release
DATASET = "q4k"            # q4k or q16k
N = 558                    # first N assemblies of MANIFEST.tsv
FORMS = "0,0 1,1"          # resident forms "packed_reference,compact_blocks"
W = "/content/panvram_run"
os.environ.update(dict(ARCHIVE_BASE_URL=ARCHIVE_BASE_URL, PANVRAM_SRC=PANVRAM_SRC, DATASET=DATASET, N=str(N), FORMS=FORMS, W=W))


In [ ]:
%%bash
# 2. panvram: download the package and build the CUDA path
set -euo pipefail
mkdir -p $W && cd $W
[ -d panvram ] || { curl -fL --retry 3 -o panvram.tar.gz "$PANVRAM_SRC"; tar -xzf panvram.tar.gz; }
cd panvram && pip install -q -e . && python -c "import panvram; assert panvram.with_cuda, 'no CUDA path'; print('panvram', panvram.__file__)"


In [ ]:
%%bash
# 3. Reference: T2T-CHM13v2.0 from NCBI, md5 of the unpacked FASTA checked
set -euo pipefail
mkdir -p $W/cohort && cd $W
if [ ! -s t2t.fa ]; then
  curl -fL --retry 3 -o t2t.fa.gz https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/009/914/755/GCA_009914755.4_T2T-CHM13v2.0/GCA_009914755.4_T2T-CHM13v2.0_genomic.fna.gz
  gunzip -c t2t.fa.gz > t2t.fa && rm -f t2t.fa.gz
fi
echo "cd1e52ce400c027ed0b7ab4b9d613f5a  t2t.fa" | md5sum -c -
ln -sf $W/t2t.fa $W/cohort/t2t.fa


In [ ]:
%%bash
# 4. Archives by the manifest: each downloaded once and its SHA-256 checked against MANIFEST.tsv
set -euo pipefail
cd $W/cohort
col=$(head -1 $W/panvram/MANIFEST.tsv | tr '\t' '\n' | grep -n "^${DATASET}_sha256$" | cut -d: -f1)
tail -n +2 $W/panvram/MANIFEST.tsv | head -n $N | cut -f1,$col > want.tsv
while IFS=$'\t' read -r name sha; do
  f=$name.$DATASET.rr3
  [ -s $f ] || curl -fsL --retry 3 -o $f "$ARCHIVE_BASE_URL/$f"
  echo "$sha  $f"
done < want.tsv > SHA256SUMS.$DATASET
sha256sum -c --quiet SHA256SUMS.$DATASET && echo "$(wc -l < want.tsv) archives, SHA-256 == MANIFEST.tsv"


In [ ]:
%%bash
# 5. On the GPU: open, sample 1024 x 8192, 1000 fetches (both against the CPU decoder), full decode of every assembly
#    with the FASTA XXH3 against the manifest; one block of C558 lines per resident form; log in $W/panvram_quickstart.txt
set -euo pipefail
ulimit -c 0
cd $W/panvram
for form in $FORMS; do
  PANVRAM_PACKED_REF=${form%,*} PANVRAM_COMPACT_BLOCKS=${form#*,} python scripts/cohort558.py $W/cohort MANIFEST.tsv $DATASET
done 2>&1 | tee $W/panvram_quickstart.txt | grep -v "^C558 full	"
